# Complainify AI — 05 : Evaluation Metrics

How accuracy, precision, recall, F1 and macro-F1 are computed on the held-out 20% test set.


---
## PART 5: EVALUATION METRICS

After training, we test on the held-out 20%:

| Metric | Formula | What it measures |
|---|---|---|
| **Accuracy** | `correct / total` | Overall correctness |
| **Precision** | `TP / (TP + FP)` | When we predict X, how often are we right? |
| **Recall** | `TP / (TP + FN)` | Of all actual X, how many did we catch? |
| **F1 Score** | `2 × P × R / (P + R)` | Harmonic mean of precision & recall |
| **Macro F1** | Average of all F1 scores | Overall performance across all categories |

In [1]:
import json, os

# Category decoder: numeric ID to human name
CAT_DECODER = {0: 'IT Support', 1: 'Hostels', 2: 'Academics', 3: 'Fees / Finance', 4: 'Maintenance', 5: 'Transport',
               6: 'Security / Discipline', 7: 'Administration', 8: 'Library', 9: 'Canteen'}

In [2]:
import sys
sys.path.insert(0, r'E:/Project-VI/workspace/ComplaintMgmtSystem/ml')
from classifier import MultinomialNB
model = MultinomialNB.load(r'E:/Project-VI/workspace/ComplaintMgmtSystem/data/model_params.json')
print('Loaded real trained model: %d classes, %d vocab' % (len(model.classes), model.vocab_size))


Loaded real trained model: 10 classes, 2324 vocab


In [3]:
# Simulate evaluation on a small test set
test_samples = [
    ('laptop charger not working', 0),          # IT Support
    ('food quality very bad in canteen', 9),    # Canteen
    ('bus is always late', 5),                  # Transport
    ('library AC not cooling', 8),              # Library
    ('water leakage in room', 1),               # Hostels
]

correct = 0
per_class = {cname: {'tp': 0, 'fp': 0, 'fn': 0} for cname in CAT_DECODER.values()}

for text, true_label in test_samples:
    pred, probs = model.predict_with_proba(text)
    true_cat = CAT_DECODER[true_label]
    pred_cat = CAT_DECODER[pred]
    
    if pred == true_label:
        correct += 1
        per_class[pred_cat]['tp'] += 1
        print(f'✓ {text:40s} → {pred_cat:20s} (CORRECT)')
    else:
        per_class[pred_cat]['fp'] += 1
        per_class[true_cat]['fn'] += 1
        print(f'✗ {text:40s} → {pred_cat:20s} (expected: {true_cat})')

print()
print(f'Accuracy: {correct}/{len(test_samples)} = {correct/len(test_samples)*100:.1f}%')
print()
print('Per-class metrics:')
print(f'{"Category":20s} {"Precision":>10s} {"Recall":>10s} {"F1":>10s}')
print('-' * 52)
macro_f1_sum = 0
for cat in sorted(per_class.keys()):
    c = per_class[cat]
    p = c['tp'] / max(c['tp'] + c['fp'], 1)
    r = c['tp'] / max(c['tp'] + c['fn'], 1)
    f1 = 2 * p * r / max(p + r, 1)
    macro_f1_sum += f1
    print(f'{cat:20s} {p:>10.4f} {r:>10.4f} {f1:>10.4f}')

print(f'{"Macro F1":20s} {"":>10s} {"":>10s} {macro_f1_sum/len(per_class):>10.4f}')

✓ laptop charger not working               → IT Support           (CORRECT)
✓ food quality very bad in canteen         → Canteen              (CORRECT)
✓ bus is always late                       → Transport            (CORRECT)
✗ library AC not cooling                   → Academics            (expected: Library)
✗ water leakage in room                    → Maintenance          (expected: Hostels)

Accuracy: 3/5 = 60.0%

Per-class metrics:
Category              Precision     Recall         F1
----------------------------------------------------
Academics                0.0000     0.0000     0.0000
Administration           0.0000     0.0000     0.0000
Canteen                  1.0000     1.0000     1.0000
Fees / Finance           0.0000     0.0000     0.0000
Hostels                  0.0000     0.0000     0.0000
IT Support               1.0000     1.0000     1.0000
Library                  0.0000     0.0000     0.0000
Maintenance              0.0000     0.0000     0.0000
Security / Discipl